# Job Listing Prioritization for Career Changers: EDA/SDA 2 (Non-Text Analysis)

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

`EDA_SDA_1.ipynb` explored the data, defended the scope decisions, and built its final table (sections 10 and 11). This notebook does the **non-text** in-depth analysis: it checks the factors kept for now, separates ghost listings from desperate employers, analyzes each time scale, narrows the target, **computer and mathematical**, to the listings a career changer can get, splits it into **role buckets** aligned with the career-switch profiles (our niche), pairs it with the team's backgrounds, and builds **its own final table**. The NLP work comes next, in `EDA_SDA_3.ipynb`; statistics in `EDA_SDA_4.ipynb`; then `TimeSeries`, `ML_Fit` and `ML_Rank`.

**Inputs from EDA_SDA_1** (decision tables in its section 10):
- English-language listings in the United States (no language label: dropped), plus listings with **no country**, kept and flagged `country_missing` for 3.8 to resolve
- Monthly horizon, April–September 2026
- Every listing's full follow-up: long-open listings kept, nothing capped
- All seniority levels and candidate factor columns, including reposts as a feature. **Only computer and mathematical listings are kept (1d, decided 2026-10-10);** every other group is cut, leaving just a per-occupation summary of pay and timing (1c)
- Career-change accessibility flags: `entry_seniority`, `years_required`, `low_experience_ask`, `entry_wording`, `changer_wording`, `degree_required`, `equivalent_experience`, and the first-pass `career_open` (EDA_SDA_1 Q1g)

**Conventions carried over:**
- Every decision is recorded with its reasoning.
- Data-quality fixes happen in cleaning; scope decisions are defended first, then applied at the end (section 4).
- Aggregates only (counts, shares, quantiles, model estimates); never `df.head()` or sample rows. Clear outputs before committing.


**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the data, and what's the scope? | Final table 1 |
| EDA_SDA_2 | Which listings do we study? The niche cut, additional cleaning, non-text checks, ghost listings | Niche table, non-text features, occupation summary |
| EDA_SDA_3 | What do the descriptions say? Text features, validated blockers, skills mapped to O*NET (no matching models) | Niche table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

**Data rules:** aggregates only, except cells tagged `data-rows` (the 2b.1 review of flagged company names): clear them before committing. Never paste rows into AI tools.

### Clarifying questions
- Which long-open listings are ghost listings, and which are employers still trying to hire?
- Does source matter beyond company hiring volume, or is it company volume in disguise?
- How precise is freshness, given the crawler finds listings a median of 1.5 days late?
- Is pay disclosure driven by state pay-transparency laws or job type, rather than employer intent?
- Does high company volume mean more competition, or more openings to win?
- How are values missing: at random, or tied to groups (source, occupation group, state)?
- How much of computer and mathematical is left once listings a career changer can't get are hidden, and is it enough for depth?
- Do high-volume employers post more entry-level listings, so that volume also signals reachability?
- Which role buckets inside computer and mathematical have enough listings (after hiding blockers) **and** enough career changers (any pool) to keep, and which are too thin?

### Roadmap

1. **Setup:** libraries and EDA_SDA_1's final table
2. **Analysis views:** which slice of the table each analysis uses, and why; **additional cleaning** (2b): staffing agencies, date consistency, internships and temporary roles, title normalization, multi-location listings
3. **Checks and analyses** (non-text)
4. **EDA_SDA_2 final table:** decisions defended, then applied at the end, with a reason for each step
5. **Conclusion and next steps** (into EDA_SDA_3)

## 1. Setup

In [ ]:
# 1a Libraries and the shared helpers (src/eda_helpers.py: paths, survival curves, tables, charts)
import gc
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, YELLOW, INK, INK_2, MUTED, CLEAN_DIR, DATA_DIR, PART1_TABLE,
    as_list, bar_h, finish, km_curve, km_median, km_survival, pct, show, surv_at, use_chart_style,
)
from src.career_switch import ROLE_ORDER, RULES, role_bucket  # the senior title rule (shared with Profiles); role buckets
from src.pl_data import JOBS_N, map_parts  # saved flattened parts, loaded one at a time
use_chart_style()

In [ ]:
# 1b Load EDA_SDA_1's final table (built and saved in its section 11): every occupation group, for 1c and 1d
if not PART1_TABLE.exists():
    raise FileNotFoundError("Run EDA_SDA_1 section 11 first: it builds and saves the final table.")
full = pd.read_pickle(PART1_TABLE)
print("EDA_SDA_1 final table:", full.shape)

In [ ]:
# 1c Occupation summary from the full table, saved before the cut (aggregates only, original listings, reposts excluded).
# It's all that's kept from outside the niche: pay and timing per occupation, for the compass's "your pay increase"
# (pay in the user's current field) and for comparing the niche with everything else.
def occ_stats(g):
    return pd.Series({
        "listings": len(g),
        "companies": g.company_id.nunique(),
        "pay disclosed": g.salary_hourly_usd.notna().mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "median days open (KM)": km_median(g.days_open, ~g.live),
        "entry-level seniority": g.entry_seniority.mean(),
        "open to career changers (any sign)": g.career_open.astype(float).mean(),
    })

originals = full[~full.is_repost]
with warnings.catch_warnings():  # occupations with no disclosed pay give an empty median: NaN, as intended
    warnings.simplefilter("ignore", RuntimeWarning)
    occupation_summary = originals.groupby(["occupation_group", "occupation"]).apply(occ_stats, include_groups=False)
    group_summary = originals.groupby("occupation_group").apply(occ_stats, include_groups=False)
OCC_SUMMARY_PATH = CLEAN_DIR / "occupation_summary.pkl"
occupation_summary.to_pickle(OCC_SUMMARY_PATH)
show("1c. Occupation groups in the full table (original listings), before the cut",
     group_summary.sort_values("listings", ascending=False).round(2))
print(f"saved {len(occupation_summary):,} occupations in {len(group_summary)} groups to {OCC_SUMMARY_PATH.name}")

In [ ]:
# 1d The cut (scope, decided 2026-10-10): keep only computer and mathematical listings from here on.
# Matched on the occupation group, or an O*NET code starting with 15- (the same family) where the group is missing.
# Credentialed math (actuaries, mathematicians) is out of scope too: credential-gated, and no career switches into it.
NICHE = "Computer and Mathematical"
by_group = full.occupation_group.eq(NICHE)
by_code = full.onet_code.fillna("").str.startswith("15-")
credentialed = full.onet_code.map(role_bucket).eq("credentialed math")
in_niche = (by_group | by_code) & ~credentialed
show("1d. The niche cut", pd.DataFrame({"listings": {
    "full table (EDA_SDA_1)": len(full),
    "computer and mathematical (kept)": in_niche.sum(),
    "  of which matched only by O*NET code": (by_code & ~by_group).sum(),
    "credentialed math (out of scope; summarized in 1c)": ((by_group | by_code) & credentialed).sum(),
    "other groups (cut; summarized in 1c)": (~in_niche).sum(),
}}))
full["company_earlier"] = full.groupby("company_id").first_seen.rank(method="min") - 1  # the company's earlier listings in any field (3.5)
company_last_new = full.groupby("company_id").first_seen.max()  # newest listing in any field, for 3.1 (one date per company)
part1 = full[in_niche].copy()
del full, originals
gc.collect()
print(f"niche table: {len(part1):,} listings from {part1.company_id.nunique():,} companies, {part1.live.sum():,} still open")

**What the data shows**
- **The niche: 14,019 computer and mathematical listings** from 6,685 companies (7.6% of 184,889), 3,458 still open. None needed the O*NET-code fallback. Credentialed math (120 listings: actuaries, mathematicians) is out of scope, and the other 170,750 listings are cut; both are kept only in the summary of 1,008 occupations in 24 groups (1c).
- **In that summary, computer and mathematical pays the most** (median $66.63 an hour, original listings) but has among the fewest entry-level listings (9%) and signs of openness to career changers (21%). Its median window, 22.5 days, is close to most groups'.

## 2. Analysis views

Some analyses are skewed by part of the table. Each analysis states which **view** it uses:

| View | Rows | Use it for | Why |
|:--|:--|:--|:--|
| **All listings** | Everything in the final table, reposts included | Ranking listings as an applicant sees them; the repost features | Each listing is a row the app could show |
| **Originals only** | Reposts (`is_repost`) excluded | Counts, volumes, application windows, company concentration | A reposted job counted twice inflates volumes and splits one long-open job into two short ones |
| **With `posted`** | About 41% of listings | Day-of-week patterns; crawler lag | `first_seen` days follow the crawler's schedule |
| **With a salary** | About 43% of listings | Pay level and pay comparisons | Pay is missing for the rest |
| **Closed listings** | Not still open at the snapshot | Simple descriptive durations | Survival methods use every listing; this view is only for plain summaries |
| **Long-open listings** | Open for a long time (threshold set in 3.1) | Ghost listings vs. desperate employers | The question only applies to them |
| **Without likely ghosts** | Every listing except `likely_ghost` (3.1d) | Every timing result (days open, share still open, open at the snapshot) is shown both with and without them, from 3.1 on | A forgotten listing's "days open" isn't a real hiring window, so ghosts can stretch timing results; they stay in the data either way |
| **Open to career changers** | `career_open`, or the stricter rule chosen in 3.11 | Accessibility comparisons within the niche; against other groups only through the 1c summary | Shows whether reachable listings behave differently |
| **The niche** | The whole table after the cut in 1d: computer and mathematical listings, by role bucket (3.6) | Everything in this notebook | Focus |
| **Other groups** | The 1c occupation summary only (aggregates; no listings) | Comparing the niche with other fields; pay in a user's current field for the compass | The listings themselves are cut |

### Confounders: compare within groups, don't filter

Some factors travel together, so a raw gap can belong to something else. **Every factor test in section 3 is checked against these,** by comparing within groups (stratifying), not by filtering them out: filtering throws data away and only answers the question for what's left. Filtering is used only when a slice can't answer the question at all (for example day of week without `posted`).

| Confounder | Why it matters here | How each check handles it |
|:--|:--|:--|
| **Company hiring volume** (earlier listings only, bands 1 / 2–4 / 5–19 / 20+) | The strongest signal; it overlaps with source, pay disclosure and staffing agencies | Every factor is compared within volume bands |
| **Source and platform** | Platforms show closures to the crawler differently | Within source type, and within platform where there are enough listings |
| **Staffing agencies** (`is_staffing`, 2b.1) | They post at volume and repost often | Gaps shown with and without them |
| **Role bucket and seniority** | A senior data scientist role and an IT support role close at different speeds for unrelated reasons | Within role bucket (3.6) where buckets are big enough; seniority as a split |
| **Month first seen** | Later months have had less time to close | Survival methods (still-open listings censored); month as a split for timing gaps |
| **Ghost listings** (`likely_ghost`, 3.1d) | Forgotten listings inflate days open and the count still open | Timing results also shown without likely ghosts |

**The rule:** each factor shows the **raw gap**, then the **gap within volume band × source type**, and within role bucket where there are enough listings. A factor counts as real only if the gap holds within the groups. EDA_SDA_4 then confirms the survivors with one model that adjusts for all of these at once.

## 2b. Additional cleaning

**Decided (2026-10-09):** five fixes and the app's filter fields here (salary sanity moved into EDA_SDA_1 6.5, description cleanup to EDA_SDA_3), run before the checks in section 3 so they use clean pay, dates and titles. Same rules as EDA_SDA_1: **fix or flag, never drop** (a flagged listing leaves only the analyses it would distort), keep the original column next to every cleaned one, and report how many listings each step changes (counts only). Spot-checks of examples are done locally by a teammate.

| # | Fix | How | Result |
|:-:|:--|:--|:--|
| 1 | **Staffing agencies** | Flag listings posted by recruiters or staffing firms: company name and domain words (staffing, recruiting, talent, well-known agency names), or an "about" text that says staffing or recruiting firm (companies table). A review cell (2b.1 review, row-level) lists the most-flagged companies so a teammate can check the keywords | `is_staffing`, kept. Report their share by field and their days open; compute company volume with and without them (3.5), since agencies post at high volume and repost often. The app labels them "posted by a recruiter" |
| – | **Salary sanity, standardized to hourly** | **Moved to EDA_SDA_1 6.5:** all pay cleaning in one place. `salary_hourly_usd` (salaried pay ÷ 2,080 hours) is the one pay column for the whole project; reversed ranges and clearly wrong units are fixed there and flagged `pay_suspect` | Arrives clean in EDA_SDA_1's table |
| 2 | **Date consistency** | EDA_SDA_1 (6.3) already removed impossible timelines and blanked `posted` dates after `first_seen`. Here: `posted` more than a year before `first_seen` | `posted_old` flag, kept; it feeds the ghost-listing check (3.1) |
| 3 | **Internships and temporary roles** | `contract_types` plus title words (intern, internship, temporary, temp, seasonal, contract-to-hire) | `job_type` (internship, temporary, contract, regular), kept: they can be entry points for career changers. Report their share in the niche and their days open; the app can filter them |
| 4 | **Title normalization** | Lowercase; strip bracketed text, locations after " - " or " \| ", requisition IDs, pay, and "remote" or "hybrid" tags (kept as a flag) | `title_clean`; the original title stays. Recheck the repost count and the role-bucket mapping with it |
| 5 | **Multi-location listings** | `n_locations` (from EDA_SDA_1) above 1 | `multi_location` flag; one row per listing for timing. The state list for the app's location filter comes with `location_data` in 3.7 |
| 6 | **Filter fields for the app** | Role level from `seniority` (entry = junior; `mid_senior` split into mid and senior by title words, the Profiles rule; manager; leadership = director and above); work mode from contract types and title; schedule from contract types | `level`, `work_mode` (remote, hybrid, onsite, not stated), `schedule` (full time, part time, not stated); with `job_type` (2b.3) and the state list (3.7) they back every filter in the app |
| – | **Description cleanup** | **Moved to EDA_SDA_3:** this table keeps only description features (`desc_words`, `desc_hash`, `has_html`), not the text, to fit in memory; EDA_SDA_3 loads the text anyway | `desc_clean` there, with HTML and boilerplate removed |

**Leakage:** every fix uses only what the listing showed when it was posted, so none of them leaks the outcome.

In [ ]:
# 2b.0 Work on a copy of EDA_SDA_1's table; every step logs how many listings it changed (counts only)
jobs = part1.copy()
clean_log = []

def log(step, change, n):
    clean_log.append({"step": step, "change": change, "listings": int(n)})

In [ ]:
# 2b.1 Staffing agencies: flag listings posted by recruiters or staffing firms (kept, flagged)
# Company name and domain come from the job openings' company records; the "about" text from the companies table.
# Both are loaded for this table's companies only, one saved part at a time.
ids = set(jobs.company_id.dropna())
names = map_parts(f"job_openings_included_{JOBS_N}",
                  lambda d: d.loc[d["id"].isin(ids), ["id", "attributes.company_name", "attributes.domain"]])
about = map_parts("companies",
                  lambda d: d.loc[d["id"].isin(ids), ["id", "attributes.description_short", "attributes.meta_title"]])
co = (names.drop_duplicates("id").set_index("id")
      .join(about.drop_duplicates("id").set_index("id"), how="outer"))

STAFFING_NAME = (r"staffing|recruit|\btalent\b|personnel|\btemps?\b|placement|headhunt|\bsearch (?:group|partners)\b"
                 r"|employment (?:agency|services|solutions)|manpower|robert half|adecco|randstad|insight global"
                 r"|teksystems|aerotek|kforce|apex systems|cybercoders|jobot|allegis|express employment|spherion"
                 r"|akkodis|beacon hill|motion recruitment|kelly services|\bhays\b")
STAFFING_ABOUT = (r"staffing (?:agency|firm|company|services|solutions)|recruit(?:ing|ment) (?:agency|firm|company)"
                  r"|executive search firm|placement (?:agency|firm)|temporary staffing|we place (?:candidates|talent)")

def text_of(*cols):
    return co.reindex(columns=list(cols)).fillna("").astype(str).agg(" ".join, axis=1).str.lower()

co["by_name"] = text_of("attributes.company_name", "attributes.domain").str.contains(STAFFING_NAME, regex=True)
co["by_about"] = text_of("attributes.description_short", "attributes.meta_title").str.contains(STAFFING_ABOUT, regex=True)
staffing_ids = co.index[co.by_name | co.by_about]
jobs["is_staffing"] = jobs.company_id.isin(staffing_ids)
log("2b.1", "flagged: posted by a staffing agency or recruiter", jobs.is_staffing.sum())

show("2b.1 Staffing agencies: companies and listings flagged", pd.DataFrame({"value": {
    "companies in this table": len(ids),
    "with a name or domain": int(co["attributes.company_name"].notna().sum()),
    "with an 'about' text": int(co["attributes.description_short"].notna().sum()),
    "flagged by name or domain": int(co.by_name.sum()),
    "flagged by 'about' text only": int((co.by_about & ~co.by_name).sum()),
    "listings flagged": int(jobs.is_staffing.sum()),
    "share of listings": round(jobs.is_staffing.mean(), 3),
}}))
by_occ = jobs.groupby("occupation").agg(listings=("id", "size"), staffing_share=("is_staffing", "mean"))
show("2b.1 Staffing share in the 10 largest computer and mathematical occupations",
     by_occ.sort_values("listings", ascending=False).head(10).round(3))
show("2b.1 Median days open (Kaplan–Meier): staffing agencies vs. employers", pd.DataFrame({
    "listings": jobs.groupby("is_staffing").size(),
    "median days open": jobs.groupby("is_staffing").apply(lambda g: km_median(g.days_open, ~g.live), include_groups=False),
}).rename(index={True: "staffing agency or recruiter", False: "employer"}))

In [ ]:
# 2b.1 review: the companies flagged most often, to check the keywords (shows company names: clear before committing)
top_flagged = jobs[jobs.is_staffing].company_id.value_counts().head(25)
display(co.loc[top_flagged.index, ["attributes.company_name", "attributes.domain", "by_name", "by_about"]]
        .assign(listings=top_flagged.values))

In [ ]:
# 2b.2 Date consistency: EDA_SDA_1 6.3 already removed impossible timelines and blanked posted dates after first_seen.
# What's left: postings dated more than a year before the crawler found them (kept, flagged; feeds the ghost check in 3.1)
jobs["posted_old"] = jobs.posted < jobs.first_seen - pd.Timedelta(days=365)
log("2b.2", "flagged posted_old: posted more than a year before first seen", jobs.posted_old.sum())

In [ ]:
# 2b.3 Internships and temporary roles: job_type from contract_types and title words (kept; the app can filter them)
title_low = jobs.title.fillna("").str.lower()
contracts = jobs.contract_types.map(as_list)
has = lambda label: contracts.map(lambda v: label in v)
JOB_TYPES = {  # first match wins
    "internship": title_low.str.contains(r"\bintern(?:ship)?s?\b|\bco-?op\b", regex=True),
    "temporary": has("temporary") | title_low.str.contains(r"\btemp(?:orary)?\b|\bseasonal\b", regex=True),
    "contract": has("contract") | title_low.str.contains(r"\bcontract(?:or|-to-hire| to hire)?\b|\bc2h\b|\b1099\b", regex=True),
}
jobs["job_type"] = np.select(list(JOB_TYPES.values()), list(JOB_TYPES), default="regular")
for kind in JOB_TYPES:
    log("2b.3", f"job_type = {kind}", (jobs.job_type == kind).sum())
show("2b.3 Job types: share of listings and median days open (Kaplan–Meier)", pd.DataFrame({
    "listings": jobs.job_type.value_counts(),
    "share": jobs.job_type.value_counts(normalize=True).round(3),
    "median days open": jobs.groupby("job_type").apply(lambda g: km_median(g.days_open, ~g.live), include_groups=False),
}))

In [ ]:
# 2b.4 Title normalization: title_clean strips tags that aren't the job (the original title stays)
REMOTE_TAG = r"\b(?:remote|hybrid|on-?site|work from home|wfh)\b"
PAY = r"\$\s?[\d,.]+\s?k?(?:\s?[-–]\s?\$?\s?[\d,.]+\s?k?)?(?:\s?(?:/|per|an?)\s?(?:hr|hour|year|yr|annually|annum))?"
REQ_ID = (r"\b(?:req(?:uisition)?|job)(?:\s*(?:id|no\.?|number))?\s*[:#]?\s*[\w-]*\d[\w-]*"
          r"|\bid\s*[:#]?\s*[\w-]*\d[\w-]*|#\s*\d+|\b\d{4,}\b")
CITY_STATE = r"^[a-z .'-]+,\s?[a-z]{2}$"

NOT_THE_JOB = rf"{CITY_STATE[1:-1]}|(?:{REMOTE_TAG[2:-2]})(?:\s+(?:us|usa))?|us|usa|united states|[\W\d]*"

def clean_title(t):
    if not isinstance(t, str):
        return t
    t = t.lower()
    t = re.sub(r"[\(\[][^\)\]]*[\)\]]", " ", t)                       # bracketed notes: location, shift, id
    parts = [p.strip() for p in re.split(r"\s+[-–|]\s+", t)]
    parts = parts[:1] + [p for p in parts[1:] if not re.fullmatch(NOT_THE_JOB, p)]  # "- Austin, TX", "| Remote", "- US"
    t = " - ".join(parts)
    for pattern in (PAY, REQ_ID, REMOTE_TAG, r"\bsign[- ]?on(?: bonus)?\b"):
        t = re.sub(pattern, " ", t)
    return re.sub(r"\s+", " ", t).strip(" -–|,:;/")

jobs["title_remote"] = title_low.str.contains(REMOTE_TAG, regex=True)
jobs["title_clean"] = jobs.title.map(clean_title)
log("2b.4", "title_clean differs from the lowercased title", (jobs.title_clean.fillna("") != jobs.title_norm.fillna("")).sum())
key = ["company_id", "title_clean", "desc_hash"]
with_key = jobs.dropna(subset=key).sort_values("first_seen")
reposts_clean = with_key.duplicated(key).sum()
show("2b.4 Title normalization", pd.DataFrame({"value": {
    "distinct titles (lowercased, EDA_SDA_1)": jobs.title_norm.nunique(),
    "distinct titles (title_clean)": jobs.title_clean.nunique(),
    "titles changed": int((jobs.title_clean.fillna("") != jobs.title_norm.fillna("")).sum()),
    "titles with a remote or hybrid tag": int(jobs.title_remote.sum()),
    "reposts (EDA_SDA_1)": int(jobs.is_repost.sum()),
    "reposts if matched on title_clean": int(reposts_clean),
}}))

In [ ]:
# 2b.5 Multi-location listings: one row each for timing; the state list for the app's filter comes with 3.7
jobs["multi_location"] = jobs.n_locations.fillna(1) > 1
log("2b.5", "flagged multi_location: names more than one location", jobs.multi_location.sum())

In [ ]:
# 2b.6 Filter fields for the app: role level, work mode, schedule (job type is 2b.3; location comes with 3.7)
# PredictLeads' seniority follows LinkedIn's labels: "mid_senior" lumps mid-level and senior together, so it's split
# with the same title rule Profiles uses for job levels (senior, sr., lead, principal, staff, architect = senior).
LEADERSHIP = ["director", "head", "vice_president", "president", "c_level", "founder", "partner"]
title_low = jobs.title.fillna("").str.lower()
seniority = jobs.seniority.fillna("not_set")
senior_title = title_low.str.contains(RULES["senior_target"], regex=True)
jobs["level"] = np.select(
    [seniority.eq("junior"), seniority.eq("mid_senior") & senior_title, seniority.eq("mid_senior"),
     seniority.eq("manager"), seniority.isin(LEADERSHIP)],
    ["entry", "senior", "mid", "manager", "leadership"], default="not stated")
log("2b.6", "level: mid_senior split into senior by title words", (seniority.eq("mid_senior") & senior_title).sum())

# Work mode from the contract-type labels, then the title; description mentions are shown as a check only
contracts = jobs.contract_types.map(as_list)
has = lambda label: contracts.map(lambda v: label in v)
title_says = lambda pattern: title_low.str.contains(pattern, regex=True)
jobs["work_mode"] = np.select(
    [has("hybrid") | title_says(r"\bhybrid\b"),
     has("remote") | title_says(r"\bremote\b|work from home|\bwfh\b"),
     has("onsite") | title_says(r"\bon-?site\b")],
    ["hybrid", "remote", "onsite"], default="not stated")
jobs["schedule"] = np.select(
    [has("full time") & has("part time"), has("full time"), has("part time")],
    ["full or part time", "full time", "part time"], default="not stated")
log("2b.6", "work_mode stated (remote, hybrid or onsite)", (jobs.work_mode != "not stated").sum())
log("2b.6", "schedule stated (full or part time)", (jobs.schedule != "not stated").sum())

def field_table(col, order):
    return pd.DataFrame({
        "listings": jobs[col].value_counts(),
        "share": jobs[col].value_counts(normalize=True).round(3),
        "median days open": jobs.groupby(col).apply(lambda g: km_median(g.days_open, ~g.live), include_groups=False),
    }).reindex([o for o in order if o in set(jobs[col])])

show("2b.6 Role level (entry = junior; mid and senior split from mid_senior by title)",
     field_table("level", ["entry", "mid", "senior", "manager", "leadership", "not stated"]))
show("2b.6 Work mode (with the share whose description mentions remote, as a check)",
     field_table("work_mode", ["remote", "hybrid", "onsite", "not stated"])
     .assign(**{"description mentions remote": jobs.groupby("work_mode").mentions_remote.mean().round(3)}))
show("2b.6 Schedule", field_table("schedule", ["full time", "part time", "full or part time", "not stated"]))

In [ ]:
# 2b.7 Summary: what each cleaning step changed (no listing removed)
show("2b. Additional cleaning: listings changed or flagged by each step", pd.DataFrame(clean_log).set_index(["step", "change"]))
print(f"listings: {len(part1):,} before, {len(jobs):,} after (fix or flag, never drop)")

**What the data shows**
- **Staffing agencies are rare in the niche:** 248 listings (1.8%) from 96 companies, and their listings close faster (median 14 vs. 22.6 days). Most flags (79 of 96 companies) come from the "about" text alone, so check the 2b.1 review cell before relying on them.
- **Role level:** entry 9.7%, mid 46.3%, senior 30.4% (split from `mid_senior` by title words), manager 10.0%, leadership 3.6%. Senior and leadership listings stay open longest (median 28 days); manager listings close fastest (17.5).
- **Work mode and schedule are missing for about half** (47.7% and 45.3%), and those listings close much faster (18 and 14 days vs. 26–31). That's most likely the source: platforms that fill in contract types may also keep listings up longer. It's a confounder to check within source type in section 3 before "not stated" is read as a signal. Where stated: remote 20.0%, hybrid 18.8%, onsite 13.6%; full time 51.3%, part time 2.5%. The description agrees: 94.6% of "remote" listings mention remote, vs. 6.4% of "not stated".
- **Job types:** 87.3% regular, 8.7% contract, 2.4% internships (337), 1.6% temporary; internships and contract roles close about as fast as regular ones (22–24 days), temporary ones faster (18).
- **Also:** 3,665 titles cleaned (11,489 distinct titles → 10,978), and matching reposts on clean titles finds the same 156; 50 postings were dated more than a year before first seen. **No listing names more than one location:** a job in several cities appears as separate listings, which the repost and cross-post split in 3.5 has to account for.

## 3. Checks and analyses

Record each result and the decision it leads to here, then apply it when building the final table (section 4). **From 3.1 on, every timing result is also shown without likely ghosts** (section 2's views). **Every factor test follows the confounders rule in section 2:** raw gap, then the gap within volume band × source type (and role bucket where possible); only gaps that hold count.

In [ ]:
# 3.0 Section 3 setup: each listing's role bucket (O*NET code; detailed 15-1299 codes first) and the counting view
jobs["role_bucket"] = jobs.onet_code.map(role_bucket).fillna("unmapped")
# Occupations hidden in the app (kept in the data): document management specialists are records work, not a tech path
HIDDEN_OCCUPATIONS = {"15-1299.03": "document management specialists"}
jobs["hidden_occupation"] = jobs.onet_code.fillna("").str[:10].isin(HIDDEN_OCCUPATIONS)
orig = jobs[~jobs.is_repost]  # counts and sizes use each job once (reposts excluded); flags stay on every listing

### 3.1 Ghost listings vs. desperate employers

A long-open listing may be a **ghost listing** (forgotten or no longer monitored: rank lower or flag it) or a **desperate employer** (still actively hiring: rank higher). Age alone can't tell them apart.

| Signal | Points to "still hiring" | Points to "ghost listing" |
|:--|:--|:--|
| Relisted (`times_listed`, `days_since_repost_last_listed`) | Relisted recently | Never relisted in months |
| Company still posting new listings | The company is actively recruiting | The company stopped posting long ago |
| Source | Hiring platforms often auto-expire listings, so a long-open one was likely kept on purpose | Company career pages can host forgotten listings indefinitely |
| `posted` vs. `first_seen` | Recently refreshed | Very old posting date |
| `last_processed_at` (meaning to confirm with PredictLeads) | May record content changes | |

**Decided (2026-10-10): hide, never remove.** **Likely ghosts** (older than **90 days** on the date being viewed and from a company gone quiet, 3.1d) are hidden in the app (a note says how many, and why); other listings older than 90 days stay visible, labelled "open 90+ days" and ranked lower; every listing stays in the analysis and the model, with still-open listings censored as usual. Age on a date is known at that moment, so hiding by it uses no future information. Removing listings by how long they *ended up* open would bias the timing model (it would never see a long window) and throw away struggling employers, who may be the most open to a career changer.

Steps: describe the long-open listings (how many, which occupation groups and sources); **check the 90-day threshold** (do closing rates level off past 90 days? are those listings relisted, or did their company stop posting?) and keep it or adjust it with that evidence; measure each signal; combine them into a **"likely ghost listing" flag**, used to rank lower or label, never to remove. Only listings first seen before July can reach 90 days in this six-month window. Evergreen wording ("always accepting applications") is added in EDA_SDA_3.

In [ ]:
# 3.1a How long do listings stay open, and does closing slow down past 90 days? (Kaplan–Meier, original listings)
# Only listings first seen in April–May can be followed for 120+ days in this window, so the curve uses them.
SNAPSHOT = pd.Timestamp("2026-09-30", tz="UTC")
STALE_DAYS = 90
early = orig[orig.first_seen < pd.Timestamp("2026-06-01", tz="UTC")]
curve = km_curve(early.days_open, ~early.live)
rows = {}
for t in [0, 30, 60, 90, 120]:
    s0, s1 = surv_at(curve, t), surv_at(curve, t + 30)
    rows[f"day {t} to {t + 30}"] = {"still open at the start": round(s0, 3),
                                    "of those, share that close in the next 30 days": round(1 - s1 / s0, 3) if s0 > 0 else np.nan}
show(f"3.1a Closing by listing age ({len(early):,} listings first seen in April–May)", pd.DataFrame(rows).T)

In [ ]:
# 3.1b Long-open listings at the snapshot: still open and more than 90 days old (labelled "open 90+ days" in the app; likely ghosts among them are hidden)
age = (SNAPSHOT - orig.first_seen).dt.days
long_open = orig.live & (age > STALE_DAYS)
show(f"3.1b Listings still open at the snapshot and older than {STALE_DAYS} days", pd.DataFrame({
    "still open": orig.groupby("role_bucket").live.sum(),
    f"older than {STALE_DAYS} days": long_open.groupby(orig.role_bucket).sum(),
    "share of open": (long_open.groupby(orig.role_bucket).sum() / orig.groupby("role_bucket").live.sum()).round(3),
}).sort_values("still open", ascending=False))
show("3.1b The same, by source type", pd.DataFrame({
    "still open": orig.groupby("source_type").live.sum(),
    "share older than 90 days": (long_open.groupby(orig.source_type).sum() / orig.groupby("source_type").live.sum()).round(3)}))

In [ ]:
# 3.1c Ghost or still hiring? Signals for open listings, long-open vs. younger (shares; all as of the snapshot)
# - company still posting: the company's newest listing in any field (saved in 1d) is from the last 60 days
# - relisted: the same job (company, clean title, description) appears more than once
# - very old posting date: posted more than a year before first seen (2b.2)
QUIET_DAYS = 60
job_key = ["company_id", "title_clean", "desc_hash"]
copies = jobs.dropna(subset=job_key).groupby(job_key).id.transform("size").reindex(jobs.index).fillna(1)
signals = pd.DataFrame({
    "company still posting": (SNAPSHOT - jobs.company_id.map(company_last_new)).dt.days <= QUIET_DAYS,
    "relisted": copies > 1,
    "on a company site or other": jobs.source_type.eq("company site / other"),
    "very old posting date": jobs.posted_old.fillna(False),
    "staffing agency": jobs.is_staffing,
})
older = orig.index[long_open]
younger = orig.index[orig.live & ~long_open]
show("3.1c Signals among open listings: older than 90 days vs. 90 days or younger", pd.DataFrame({
    f"older than {STALE_DAYS} days ({len(older):,})": signals.loc[older].mean().round(3),
    f"{STALE_DAYS} days or younger ({len(younger):,})": signals.loc[younger].mean().round(3),
}))

In [ ]:
# 3.1d A first ghost-listing flag (provisional; evergreen wording joins it in EDA_SDA_3). Used to label or rank lower,
# never to remove. Among listings open and older than 90 days:
# - likely ghost: the company has posted nothing new in 60 days, or the posting date is more than a year old
# - possibly hard to fill: the company is still posting. Tested below and dropped as a category on 2026-10-10: these
#   listings are no more open to newcomers than younger ones, so stale listings that aren't likely ghosts are just "stale"
jobs["age_at_snapshot"] = (SNAPSHOT - jobs.first_seen).dt.days
jobs["stale"] = jobs.live & (jobs.age_at_snapshot > STALE_DAYS)
quiet = ~signals["company still posting"]
jobs["likely_ghost"] = jobs.stale & (quiet | signals["very old posting date"])
jobs["possibly_hard_to_fill"] = jobs.stale & ~jobs.likely_ghost
o = jobs.loc[orig.index]
show("3.1d Long-open listings: likely ghosts vs. possibly hard to fill (original listings)", pd.DataFrame({
    "stale (open, older than 90 days)": o.groupby("role_bucket").stale.sum(),
    "likely ghost": o.groupby("role_bucket").likely_ghost.sum(),
    "possibly hard to fill": o.groupby("role_bucket").possibly_hard_to_fill.sum(),
}).sort_values("stale (open, older than 90 days)", ascending=False))
print(f"stale: {o.stale.sum():,} of {o.live.sum():,} open listings | likely ghosts: {o.likely_ghost.sum():,} | "
      f"possibly hard to fill: {o.possibly_hard_to_fill.sum():,}")

# Is "possibly hard to fill" worth keeping? Only if those listings are more open to newcomers than the rest
o = o.assign(group=np.select([o.likely_ghost, o.possibly_hard_to_fill, o.live],
                             ["likely ghost", "possibly hard to fill", "open, 90 days or younger"], default="closed"),
             is_entry=o["level"].eq("entry"), is_open_to_changers=o["career_open"].astype(float))
show("3.1d Openness to career changers: possibly hard to fill vs. likely ghosts vs. younger open listings",
     o[o["group"] != "closed"].groupby("group").agg(listings=("group", "size"), entry_level=("is_entry", "mean"),
                                                   open_to_career_changers=("is_open_to_changers", "mean")).round(3))

**What the data shows**
- **90 days is a reasonable stale line, but not a sharp break.** Among listings first seen in April–May, the share of open listings that close within the next 30 days falls steadily: 54% in the first month, 42%, 32%, then 26% from day 90 to 120. A 90-day-old listing is about half as likely to close next month as a new one, yet a quarter still close each month, so many stale listings aren't dead; the ghost flag tells them apart.
- **666 of 3,447 open listings (19%) are more than 90 days old,** from 14% (quality assurance) to 22% (software development). Every open job-board listing (47) is more than 90 days old, likely how job boards keep listings up; 3.2 looks at it.
- **One signal separates ghosts from active listings:** only 44% of stale listings come from a company that posted anything new in the last 60 days, vs. 92% of younger ones. Relisting, very old posting dates and source barely differ, so the flag rests almost entirely on company activity.
- **374 likely ghosts (56% of stale);** without them, median days open drops 1–3 days per bucket and open counts about 10% (3.12a).
- **"Possibly hard to fill" is dropped:** those 292 stale listings from still-active companies are no more open to career changers than younger listings (19.6% vs. 20.0% show any sign; 7.9% vs. 10.2% entry level), so the idea that a struggling employer welcomes newcomers doesn't hold here. Stale listings that aren't likely ghosts are just "stale". **In the app, only likely ghosts are hidden;** other stale listings stay visible, labelled "open 90+ days" and ranked lower, since a quarter of 90-day-old listings still close each month (3.1a). Likely ghosts are clearly less open (14.5%).

### 3.2 Source / platform

| Check | If the result is… | Then… |
|:--|:--|:--|
| Median days open by source, *within* each company-volume band | Source gaps hold within every band | Keep source as its own factor |
| | Gaps mostly disappear | Drop it, or keep it as a minor factor |
| Share of job-board listings whose company + title also appears on a company site or platform | High (copies) | Drop job-board rows to avoid double-counting |
| | Low | Keep them |
| Platform sizes in the chosen groups | Only a few platforms have 1,000+ listings | Name those platforms; group the rest as "other platform" |

### 3.3 Freshness and crawler lag

| Check | If the result is… | Then… |
|:--|:--|:--|
| Share of listings found more than 7, 14 and 30 days after `posted` | Tiny | Use `first_seen` only |
| | Meaningful | Add a "found late" flag |
| Survival curves: found late vs. found promptly | Found-late listings close much sooner | The flag matters for apply-by estimates |
| Survival curves: listings with vs. without a `posted` date | Similar | `posted` can be used where present |
| | Different | Stick with `first_seen` only |

### 3.4 Pay disclosed

| Check | If the result is… | Then… |
|:--|:--|:--|
| Disclosure rate by US state (needs the state from `location_data`) | Much higher in pay-transparency states (CA, CO, NY, WA) | Add state, or a "pay-law state" flag |
| Disclosure by hourly vs. salaried, and by seniority | Strongly tied to hourly | Its effect may really be job type |
| Days open by disclosure, within company-volume bands | The gap holds | Keep as a factor |
| | The gap disappears | Keep only as the companion flag for pay level |

### 3.5 Reposts and company hiring volume

- **Cross-posted vs. reposted (added 2026-10-10).** EDA_SDA_1 counts every later copy of the same job (same company, title and description) as a repost, so a copy on a second board while the first is still up looks the same as a job relisted after it closed. Split them, using each listing's earlier copies only, so nothing comes from the future:
  - **`is_crosspost`:** an earlier copy is still live when this one appears, from a different source (the platform, or "company site" or "job board" when there's no platform).
  - **`is_relisted`:** earlier copies exist, but none is still live: the job came back after closing.
  - **`n_sources_so_far`:** how many different sources carry the job at the moment this listing appears (1 = only here).
  - **Hypothesis: posting on several boards signals urgency.** Test it, don't assume it: days open (Kaplan–Meier) by `n_sources_so_far` and cross-posted vs. relisted vs. single, within company-volume bands and with and without staffing agencies (2b.1). Applicant-tracking systems often send one posting to several boards automatically, so check within platform too: if the gap disappears there, it's the software, not the employer's urgency.
  - **Copies in different cities aren't cross-posts:** each listing names one location (2b.5), so a job in several cities appears as several listings. The table has no per-listing city yet, so **source stands in for it:** a live earlier copy on a different source is a cross-post; on the same source, most likely another city (`is_same_source_copy`).
  - **Matching (decided after the first run, 2026-10-10):** copies are matched on company and clean title, not the description: boards reformat descriptions, and the strict match found 0 cross-posts (a looser check found 460 same-title copies within 7 days).
- **Reposts** (a decided feature: how strongly should it count?): days open by `times_listed` and by relisted vs. cross-posted; repost rate in the chosen groups; repost rate within company-volume bands, to see whether it adds anything beyond company volume.
- **Company hiring volume:** does high volume mean **more competition, or more openings to win?** Its direction in the ranking depends on the answer. Count each company's listings from before each listing only, so the future isn't used. The companies table has no company size or employee count, so listing volume is our only measure of how much a company hires.
- **For career changers:** do high-volume employers post a larger share of entry-level listings or training wording? If so, volume also signals that a newcomer can get in.

In [ ]:
# 3.5a Company hiring volume, counted from each company's EARLIER listings in any field (1d), the way the model sees it.
# Bands count the listing itself: 1 / 2-4 / 5-19 / 20+. Raw gap, then within source type (the confounders rule),
# with and without staffing agencies, and without likely ghosts (3.1d).
VOL_BANDS = ["1 posting", "2-4", "5-19", "20+"]
jobs["volume_band"] = pd.cut(jobs.company_earlier + 1, [0, 1, 4, 19, np.inf], labels=VOL_BANDS)
o = jobs.loc[orig.index]

def km_by(df, by):
    return df.groupby(by, observed=True).apply(lambda g: km_median(g.days_open, ~g.live), include_groups=False)

show("3.5a Median days open (KM) by company volume (earlier listings only)", pd.DataFrame({
    "share of listings": o.volume_band.value_counts(normalize=True).round(3),
    "all listings": km_by(o, "volume_band"),
    "without staffing agencies": km_by(o[~o.is_staffing], "volume_band"),
    "without likely ghosts": km_by(o[~o.likely_ghost], "volume_band"),
    "entry level": o["level"].eq("entry").groupby(o["volume_band"], observed=True).mean().round(3),
}).reindex(VOL_BANDS))
show("3.5a The same within source type (median days open, KM)",
     km_by(o, ["source_type", "volume_band"]).unstack("volume_band").reindex(columns=VOL_BANDS).round(1))

In [ ]:
# 3.5b Copies of the same job (same company and clean title): cross-posted, same-source copy, or relisted.
# Matched without the description: boards reformat descriptions, so a strict match found no cross-posts at all.
# A company with several open roles under one title can look like copies; requiring overlapping live dates keeps that in check.
# Uses only each listing's EARLIER copies and whether they were still live when it appeared (known at that moment).
# The table has no per-listing city, so source stands in for it: a live earlier copy on a DIFFERENT source is a
# cross-post (another board); on the SAME source it's most likely the same job in another city.
jobs["source"] = jobs.platform.fillna(jobs.source_type)
key = ["company_id", "title_clean"]
keyed = jobs.dropna(subset=key)
multi = keyed[keyed.groupby(key).id.transform("size") > 1][key + ["id", "first_seen", "last_seen", "source"]]
pairs = multi.merge(multi, on=key, suffixes=("", "_earlier"))
pairs = pairs[pairs.first_seen_earlier < pairs.first_seen]                      # earlier copies only
pairs["live_then"] = pairs.last_seen_earlier >= pairs.first_seen                # still up when this one appeared
pairs["other_source"] = pairs.source_earlier != pairs.source
per = pairs.groupby("id").agg(
    earlier_copies=("id_earlier", "size"),
    live_other_source=("other_source", lambda s: (s & pairs.loc[s.index, "live_then"]).any()),
    live_same_source=("other_source", lambda s: (~s & pairs.loc[s.index, "live_then"]).any()),
    live_other_sources=("source_earlier", lambda s: pairs.loc[s.index][pairs.loc[s.index, "live_then"]
                                                                      & pairs.loc[s.index, "other_source"]].source_earlier.nunique()))
jobs = jobs.join(per, on="id")
jobs[["earlier_copies", "live_other_sources"]] = jobs[["earlier_copies", "live_other_sources"]].fillna(0).astype(int)
jobs["is_crosspost"] = jobs.live_other_source.fillna(False).astype(bool)
jobs["is_same_source_copy"] = ~jobs.is_crosspost & jobs.live_same_source.fillna(False).astype(bool)
jobs["is_relisted"] = (jobs.earlier_copies > 0) & ~jobs.is_crosspost & ~jobs.is_same_source_copy
jobs["n_sources_so_far"] = 1 + jobs.live_other_sources  # its own source plus other sources carrying the job when it appeared
jobs["copy_type"] = np.select([jobs.is_crosspost, jobs.is_same_source_copy, jobs.is_relisted],
                              ["cross-posted", "same-source copy", "relisted"], default="single")
log("3.5", "cross-posted (another source had the job live)", jobs.is_crosspost.sum())
log("3.5", "relisted (earlier copies, none live)", jobs.is_relisted.sum())

# Looser match: same company and clean title, first seen within 7 days (catches boards that reformat descriptions)
o = jobs.loc[orig.index]
show("3.5b Copies of the same job", pd.DataFrame({"listings": {
    "single (no earlier copy)": (jobs.copy_type == "single").sum(),
    "cross-posted (another source live)": jobs.is_crosspost.sum(),
    "same-source copy (likely another city)": jobs.is_same_source_copy.sum(),
    "relisted (earlier copies closed)": jobs.is_relisted.sum(),
}}))

In [ ]:
# 3.5c Does posting on several boards signal urgency? Days open by number of sources and copy type, overall and
# within the listing's own platform (if the gap disappears there, it's the hiring software, not the employer).
# Uses every listing (copies included), since the copies are the point; still-open listings are censored.
jobs["sources_band"] = pd.cut(jobs.n_sources_so_far, [0, 1, 2, np.inf], labels=["1", "2", "3+"])
show("3.5c Median days open (KM) by copy type", pd.DataFrame({
    "listings": jobs.copy_type.value_counts(),
    "median days open": km_by(jobs, "copy_type"),
    "without likely ghosts": km_by(jobs[~jobs.likely_ghost], "copy_type"),
}))
show("3.5c Median days open (KM) by sources carrying the job, within source type",
     km_by(jobs, ["source_type", "sources_band"]).unstack("sources_band").round(1))

**What the data shows**
- **Company hiring volume is the strongest signal in the niche:** listings from companies with 20+ earlier listings (any field) close in a median **7.1 days** vs. **38.9** for a company's first listing, about 5.5× (EDA_SDA_1 found 2.3× across all fields). It holds without staffing agencies (7.1 vs. 39.0) and without likely ghosts (7.1 vs. 33.8), and within every source type: company sites 6.1 vs. 48.1 days, hiring platforms 18.2 vs. 42.1; job boards show a weaker gap (12.0 vs. 16.5).
- **Big employers aren't more open to newcomers:** 8–11% of listings are entry level in every volume band, so volume is about speed, not access.
- **To check in 3.3:** companies with many listings may be crawled more often, which would shorten the windows we observe for them.
- *To fill in after the rerun of 3.5b–c (copies are now matched on company and clean title, since the strict description match found no cross-posts): how many copies are cross-posted, same-source or relisted, and whether multi-board posting goes with faster closing within source type.*

### 3.6 The niche: computer and mathematical, by role bucket

**Decided (2026-10-09; applied at the start, in 1d, since 2026-10-10):** the niche is **computer and mathematical listings in the US**, with listings a career changer can't get hidden (3.11). It isn't narrowed to one role: role discovery needs several roles to choose from, and a single role would leave too few listings and too few career-switch examples. Instead, **role bucket** is a dimension inside the niche, defined the same way on both sides, so the listings (what's open) and the career changers in `Profiles.ipynb` (who got there) line up role by role:

| Role bucket | Listings: O*NET codes (2019 SOC) | Profiles: resume titles |
|:--|:--|:--|
| data and analytics | 15-2051 data scientists, 15-2041 statisticians, 15-2031 operations research, 15-1221 research scientists, 15-2099 | data analyst / scientist / engineer, BI, statistician, machine learning |
| database | 15-1242, 15-1243 | database administrator or developer, SQL developer |
| quality assurance | 15-1253 | QA analyst or tester |
| security | 15-1212 | information, cyber or IT security analyst |
| networks and systems | 15-1211, 15-1231, 15-1241, 15-1244 | network or systems administrator, engineer or analyst; DevOps; cloud |
| it support | 15-1232 | help desk, desktop support, IT support |
| software development | 15-1251, 15-1252, 15-1254, 15-1255 | software or web developer, programmer |
| other computer roles | 15-1299 | (none) |
| credentialed math | 15-2011 actuaries, 15-2021 mathematicians | (none) |

The listing side is `role_bucket(onet_code)` (`ONET_DETAILED_BUCKETS` for detailed 15-1299 codes first, then `ONET_ROLE_BUCKETS`, which also covers the older 2010 codes) and the resume side is `RULES["cm_buckets"]`, both in `src/career_switch.py`.

**Steps:**
1. **Map** every computer and mathematical listing to a role bucket from `onet_code`, and report unmapped codes (aggregate counts) so the mapping can be fixed.
2. **Size each bucket** on originals only: listings, listings still open, companies and the top company's share, with and without manager-level listings (3.11; they're hidden per applicant, so the niche keeps them).
3. **Count career changers per bucket** from `Profiles.ipynb`'s modeling table (`career_changer`, by the role at the switch; unique people, after duplicate removal), split by pool: retrained after working, retrained as new grads, and changed roles within tech.
4. **Depth check per bucket:** measure the 95% confidence-interval width of the Kaplan–Meier share still open after 7 days in each sub-group. Enough depth means **±5 points or narrower** (a width of 10 or less), tight enough to show the 5–16 point gaps the EDA found, in at least 80% of sub-groups. Splits are tried from finest to coarsest, combining sub-groups that are too wide: month × source type, then source type, then month, then the whole bucket. The finest split that passes is how finely that bucket is analyzed in section 3. (About 300+ listings per sub-group are needed for ±5, so most buckets won't support all 18 month × source sub-groups.)
5. **Keep a bucket** only if it passes the depth check on the listings side **and** has enough career changers to learn from (at least 30 unique people, counting **all three pools**). The 30 was first set for retrained people only, before the within-tech pool existed; counting retrained only would drop data and analytics (22 retrained, 238 in all), which the compass needs to show is reached through other tech roles. Thin buckets are combined with a close one or analyzed only as part of the niche; their listings always stay in the table. **Credentialed math** (actuaries, mathematicians) is out of the niche from 1d: credential-gated, with no career switches into it. **IT project managers** (O*NET 15-1299.09, about 13% of the niche) have their own bucket since 2026-10-10, on both sides (a Profiles rules change). Record each keep, combine or drop with its reason.
6. **Save the niche** (kept buckets, blockers hidden) as its own table, the base for section 4 and EDA_SDA_3.

**Also described per bucket** (not used to choose buckets): variation in the factors (window, source, company volume, reposts, pay), coverage of key fields (description, salary, `posted`), words per description (EDA_SDA_1 Q3e), and openness to career changers (`entry_share`, `career_open_share`, years asked, degree required).

**Related groups:** the team's source-background groups are cut with the rest (1d); they live on as rows of the 1c occupation summary (pay, timing, openness) and through O*NET skills and the Career Changers Matrix for role discovery.

In [ ]:
# 3.6a What's in each role bucket (mapped in 3.0 from the O*NET code; detailed 15-1299 codes first)
other = orig[orig.role_bucket.isin(["other computer roles", "unmapped"])]
show("3.6a What's left in 'other computer roles' and unmapped, by occupation (original listings)",
     other.groupby(["role_bucket", "occupation"]).size().sort_values(ascending=False).head(15).to_frame("listings"))
print(f"IT project managers (O*NET 15-1299.09): {orig.onet_code.fillna('').str.startswith('15-1299.09').sum():,} original listings")
print(f"hidden in the app (document management specialists): {orig.hidden_occupation.sum():,} original listings")

In [ ]:
# 3.6b Size each bucket (original listings), with and without manager and leadership listings (hidden per applicant)
def bucket_size(g):
    top_share = g.company_id.value_counts(normalize=True).iloc[0]
    return pd.Series({
        "listings": len(g),
        "without manager or leadership": (~g.level.isin(["manager", "leadership"])).sum(),
        "still open": g.live.sum(),
        "companies": g.company_id.nunique(),
        "top company's share": round(top_share, 3),
        "entry level": round((g.level == "entry").mean(), 3),
        "median days open (KM)": km_median(g.days_open, ~g.live),
        "median hourly pay": g.salary_hourly_usd.median(),
    })

sizes = orig.groupby("role_bucket").apply(bucket_size, include_groups=False).sort_values("listings", ascending=False)
show("3.6b Role buckets in the niche (original listings)", sizes.round(2))

In [ ]:
# 3.6c Career changers per bucket, by pool (Profiles' frozen modeling table; counts only)
profiles = pd.read_pickle(DATA_DIR / "profiles" / "modeling_table.pkl")
changers = profiles[profiles.career_changer]
by_pool = pd.crosstab(changers.target_bucket, changers.pool, margins=True, margins_name="all pools")
show("3.6c Career changers per role bucket, by pool (Profiles)", by_pool)

In [ ]:
# 3.6d Depth check: how finely can each bucket be split and still give precise results?
# Measure: Kaplan–Meier share still open after 7 days, with its 95% interval. Precise enough = an interval of
# 10 points or less (±5), tight enough for the 5-16 point gaps the EDA found. A split is deep enough if at least 80%
# of its sub-groups are that tight. Splits are tried from finest to coarsest, combining sub-groups as the plan says:
# month × source type, then source type only, then month only, then the whole bucket. (Rules fixed before looking.)
from lifelines import KaplanMeierFitter
MAX_WIDTH, MIN_PASS_SHARE = 0.10, 0.80
SPLITS = {"month × source type": ["month", "source_type"], "source type": ["source_type"], "month": ["month"],
          "whole bucket": []}

def ci_width_at_7(g):
    if len(g) < 2:
        return np.nan
    kmf = KaplanMeierFitter().fit(g.days_open, event_observed=~g.live)
    ci = kmf.confidence_interval_survival_function_
    row = ci[ci.index <= 7].iloc[-1]
    return float(row.iloc[1] - row.iloc[0])

full_months = orig[~orig.partial_month]
rows = {}
for bucket, g in full_months.groupby("role_bucket"):
    found = None
    for name, cols in SPLITS.items():
        widths = g.groupby(cols).apply(ci_width_at_7, include_groups=False) if cols else pd.Series([ci_width_at_7(g)])
        if (widths <= MAX_WIDTH).mean() >= MIN_PASS_SHARE:
            found = (name, len(widths), widths.median())
            break
    rows[bucket] = {"listings (full months)": len(g),
                    "finest deep-enough split": found[0] if found else "none",
                    "sub-groups": found[1] if found else 0,
                    "median interval width": round(found[2], 3) if found else np.nan}
depth_summary = pd.DataFrame(rows).T
depth_summary["listings side passes"] = depth_summary["finest deep-enough split"] != "none"
show("3.6d Depth check: the finest split where 80% of sub-groups have a 7-day interval of 10 points or less",
     depth_summary.sort_values("listings (full months)", ascending=False))

In [ ]:
# 3.6e Which buckets get their own analysis (listings are never removed; this only decides how they're analyzed): the listings side must pass the depth check AND the bucket needs
# at least 30 career changers across all three pools (both thresholds fixed before looking at results)
MIN_CHANGERS = 30
decision = pd.DataFrame({
    "listings": sizes["listings"],
    "listings side passes": depth_summary["listings side passes"],
    "career changers": by_pool["all pools"].drop("all pools", errors="ignore"),
}).fillna({"listings side passes": False, "career changers": 0})
decision["enough career changers"] = decision["career changers"] >= MIN_CHANGERS
decision["verdict"] = np.select(
    [decision["listings side passes"] & decision["enough career changers"],
     decision["listings side passes"] | decision["enough career changers"]],
    ["own analysis and compass paths", "combine with a close bucket"], default="no separate analysis")
show("3.6e Role buckets: own analysis, combine, or none (every listing stays in the niche table either way)",
     decision.sort_values("listings", ascending=False))

**What the data shows**
- **Eight buckets get their own analysis and compass paths:** software development (4,572 listings, 475 career changers), networks and systems (2,724; 670), IT project management (1,832; 264), data and analytics (1,739; 237), security (744; 350), IT support (711; 383), database (698; 395) and quality assurance (546; 39, just over the 30 needed).
- **No bucket supports month × source type:** software development and networks and systems can be split by month; the rest are analyzed as a whole bucket. Section 3's checks use each bucket's finest deep-enough split.
- **IT support is the clearest entry door:** the most entry-level listings (22%) and the fastest closing (median 18.3 days), but by far the lowest pay ($32.23 an hour vs. $61–72 for the others). Quality assurance is next most open (14% entry level).
- **IT project management is the hardest to enter:** 2% entry level and 55% manager or leadership listings, with pay tied for highest ($72.32 an hour). Of its 264 career changers, 190 moved in from another tech role, so it's mostly a later move, not a first step; the compass shows it that way.
- **"Other computer roles" is down to 298 listings** (GIS technologists 163, document management 70, others 65): no matching career switches, so no separate analysis; its listings stay in the table.

### 3.7 Blind spots from Part 1

| Blind spot | Check or fix |
|:--|:--|
| The "closed" outcome is wrong for reposted jobs: the earlier listing disappears without being filled | In the analysis (not as a model input), treat a listing that was later reposted as "closed without filling" or censored |
| The 2-day "still open" tolerance is a guess | Distribution of days between `last_seen` and the snapshot; a bump near 7 days means it's too tight |
| Converting salaried pay to hourly assumes full time (2,080 hours) | Hourly listings keep their posted rate, so only salaried part-time listings are off; flag them with `contract_types` (part time) |
| Company volume counted within the sample | The companies table has no size or employee count. It does have a description (industry, via NLP in EDA_SDA_3), a ticker (publicly traded), parent and subsidiary companies, and similar companies: check whether these add anything beyond volume |
| State isn't in the table | Add it from `location_data`, if PredictLeads includes it |
| Missing country vs. remote, missing language vs. description | Confirm the EDA_SDA_1 section 10a results |
| `entry_seniority` counts only the `junior` label | Check every seniority label (EDA_SDA_1 Q3b) and add any that are entry level, such as intern, to `ENTRY_LEVELS` |

### 3.8 Joining the companies table

Merge the companies table into the listings (a **left join** on `company_id` = company `id`, so every listing stays). Course standard: one master table from all sources, with keys checked first. Join checks: company ids are unique in the companies table (no row multiplication), and match rate (100% in EDA_SDA_1). Columns to bring in: `ticker` (publicly traded), parent company, subsidiary and similar companies, location, and description (its text goes to EDA_SDA_3). The table has **no company size or employee count**.

**Then resolve listings with no country** (`country_missing`, kept by EDA_SDA_1): take the country from the company's `location_data`. Keep the listing if the company is US-based **or** the listing mentions remote work or uses the "remote" contract type, and flag it `country_inferred`; drop the rest. Report how many are kept by each route and how many are dropped, and check whether `country_inferred` listings behave differently (days open, source, words per description).

### 3.9 Missing-data patterns

**First classify each field** (structural, censored, or missing for a reason), then treat it. Most of ours aren't truly missing, so very little is imputed.

| Field | Missing (final table) | Why | Handling |
|:--|:--|:--|:--|
| `first_seen`, `last_seen`, title, seniority, occupation group, `onet_code` | ~0% | n/a | Nothing. A listing with no `onet_code` can't get a role bucket, so it's outside the niche (report the count) |
| `last_seen` of **live** listings | 23% still open | **Censored,** not missing | Never imputed or dropped; survival methods handle it |
| `platform` | 60.7% | **Structural:** company sites have no platform | Not missing; `source_type` says "company site / other" |
| `status` | 35% | Never says "open" | Not used; `last_seen` decides what's live |
| `posted` | 55% | Many sources publish no date | **Not imputed.** Freshness uses `first_seen`; day of week uses the "with `posted`" view; add `has_posted`, and check whether listings with and without it differ |
| Salary | 53% | **Missing for a reason** (pay-transparency laws, hourly vs. salaried) | **Pay level isn't filled in.** `salary_disclosed` stays its own factor. Trees use blanks natively; linear models get a median by role bucket × seniority **plus** a missing flag, fit inside the `Pipeline` on training months only. The app shows "not disclosed" |
| Description and its text flags | 9.8% | Some sources expose no text | Listing kept; text flags are **unknown** (blank), not False, since EDA_SDA_1's fix; `has_desc` is the flag |
| `years_required` | Blank when no years are stated | **"Not stated" isn't zero** | Kept blank, with a "states years" flag; not stated never counts as a blocker |
| Repost features | Blank without a description | Can't be matched | Kept blank, not 0 (0 would claim "first listing") |
| `tags` | 53% empty | Sparse | Not used; `categories` (99%) instead |
| Company description, location (joined in 3.8) | 10%, 6% | Not in the companies file | "Unknown" category |

**Course standard:** percent missing, missingness **by group** (source, occupation group, role bucket, month, state), and `df.isnull().corr()` to see which fields go missing together. Every imputation or "Unknown" category is justified with plots split by the outcome (closed vs. still open). **Rules:** impute only inside the model `Pipeline`, fit on the training months (leakage rules in `HOW_WE_WORK.md`), and add a missing flag whenever a value is filled in, since missingness itself can carry signal (salary disclosure does).

### 3.10 Outliers

Boxplots and IQR rules for days open, words per description, hourly pay, times listed. Course standard: inspect, explain, and keep or remove with a reason. Advanced: robust (median / IQR) summaries, and log scales for the right-skewed measures.

### 3.11 Career-change accessibility, and hiding what a career changer can't get

**Decided (2026-10-09): listings a career changer can't get are hidden.** Three kinds of blocker, each hidden where it's reliable:

| Blocker | Example | Hidden in |
|:--|:--|:--|
| Seniority, per applicant | Manager level and above (`manager`, `head`, `director`, `vice_president`, `president`, `c_level`, `partner`, `founder`; from EDA_SDA_1 Q3b) for applicants without management or senior experience | The app, per profile; this notebook's table keeps them, labelled |
| Text, for everyone | Too many years of experience asked; a required license | EDA_SDA_3's final table, after the rules are validated |
| Text, per applicant | A degree or certification this applicant doesn't have | The app, per profile |
| Occupation, for everyone | Document management specialists (O*NET 15-1299.03): records work, not a tech path (decided 2026-10-10) | The app; flagged `hidden_occupation` in 3.0 and kept in the table |

Report how many listings each rule hides. Then check:

| Check | If the result is… | Then… |
|:--|:--|:--|
| Accessibility flags by company-volume band and by source | Entry-level listings cluster at high-volume employers or on certain platforms | Account for it, so accessibility isn't company size or source in disguise |
| Pay of listings open to career changers vs. the rest, within the niche | A large pay gap | Show it in the app, and let the pay weight trade off against accessibility |
| Days open by accessibility, within the niche | Reachable listings close faster | Raise their urgency in the ranking |
| Degree required by role bucket | Near-universal in a bucket, with little "or equivalent experience" | That bucket is a poor target |
| Each part of `career_open` separately | One part (for example, entry-level wording) adds little | Use a stricter rule for the "open to career changers" view |

### 3.12 The retraining compass: the 2026 side

For each kept role bucket, after blockers are hidden: **open listings at the snapshot, new listings per week, median hourly pay (where disclosed), median days open (Kaplan–Meier), entry-level share, openness to career changers, remote share, and expected job growth** (a placeholder column until the projections are downloaded) (official projections by occupation code; time frame to be decided). Rows in the same role order as Profiles (`ROLE_ORDER` in `src/career_switch.py`: infrastructure → code → data).

Join it by role bucket to Profiles' three path tables, so every path shows its 2026 openings, pay, closing speed and growth:
- **Prior career → first tech job** and **field trained in → first tech job** (4a.2b): the compass for changing careers.
- **Tech job before → after the move** (4a.3a), with the sideways-or-up split: the compass for moving across or up within tech.

Every path carries its people count, and paths under about 10 people are marked thin. Growth sits next to the entry-level share, since projections cover the whole occupation. Aggregates only; this table feeds the app's Retraining compass and the deck (`DIFFERENTIATORS.md`).

In [ ]:
# 3.12a The 2026 market for each role bucket (original listings; aggregates only), with timing also shown without likely ghosts
full_weeks = orig.loc[~orig.partial_month, "first_seen"]
WEEKS = max((full_weeks.max() - full_weeks.min()).days / 7, 1)

def market(g):
    return pd.Series({
        "open at the snapshot": int(g.live.sum()),
        "new listings per week": round((~g.partial_month).sum() / WEEKS, 1),
        "median hourly pay": g.salary_hourly_usd.median(),
        "median days open (KM)": km_median(g.days_open, ~g.live),
        "entry level": round((g.level == "entry").mean(), 3),
        "open to career changers (any sign)": round(g.career_open.astype(float).mean(), 3),
        "remote": round((g.work_mode == "remote").mean(), 3),
        "expected growth": np.nan,  # filled once the official occupational projections are downloaded (queued)
    })

o = jobs.loc[orig.index]  # original listings, with the 3.1d ghost flag
role_market = o[o.role_bucket.isin(ROLE_ORDER)].groupby("role_bucket").apply(market, include_groups=False)
no_ghosts = o[o.role_bucket.isin(ROLE_ORDER) & ~o.likely_ghost]
role_market["open, without likely ghosts"] = no_ghosts.groupby("role_bucket").live.sum()
role_market["median days open, without likely ghosts"] = no_ghosts.groupby("role_bucket").apply(
    lambda g: km_median(g.days_open, ~g.live), include_groups=False)
role_market = role_market.reindex([r for r in ROLE_ORDER if r in role_market.index])
show("3.12a The 2026 market by role bucket (original listings)", role_market)

In [ ]:
# 3.12b Join the market to Profiles' paths (from the frozen modeling table; counts only). Paths under 10 people are thin.
THIN = 10
retrained = changers[changers.pool.str.startswith("retrained")]
within = changers[changers.pool.eq("changed roles within tech")]
paths = pd.concat([
    retrained.groupby(["background_area", "target_bucket"]).size().rename("people").reset_index()
        .rename(columns={"background_area": "from"}).assign(path="prior career"),
    retrained.groupby(["retrain_field", "target_bucket"]).size().rename("people").reset_index()
        .rename(columns={"retrain_field": "from"}).assign(path="field trained in"),
    within.groupby(["from_bucket", "target_bucket"]).size().rename("people").reset_index()
        .rename(columns={"from_bucket": "from"}).assign(path="tech job before the move"),
], ignore_index=True)
paths["thin (under 10 people)"] = paths.people < THIN
compass_2026 = paths.merge(role_market, left_on="target_bucket", right_index=True, how="left")

for kind in ["prior career", "field trained in", "tech job before the move"]:
    top = compass_2026[compass_2026.path == kind].sort_values("people", ascending=False).head(8)
    show(f"3.12b Largest paths by {kind}, with the 2026 market of the role they lead to",
         top.set_index(["from", "target_bucket"])[["people", "thin (under 10 people)", "open at the snapshot",
                                                   "median hourly pay", "median days open (KM)", "entry level"]])
share_thin = compass_2026.groupby("path")["thin (under 10 people)"].mean().round(3)
show("3.12b Share of paths resting on fewer than 10 people", share_thin.to_frame("share thin"))

In [ ]:
# 3.12c Save the 2026 side for ML_Fit, which assembles the compass and the Ideas tab (aggregates only)
role_market.to_pickle(CLEAN_DIR / "role_market.pkl")
compass_2026.to_pickle(CLEAN_DIR / "compass_paths_2026.pkl")
print(f"saved role_market.pkl ({len(role_market)} buckets) and compass_paths_2026.pkl ({len(compass_2026):,} paths)")

**What the data shows**
- **The 2026 market by bucket:** software development is the largest (1,175 open at the snapshot, about 174 new a week), then networks and systems (676; 103), IT project management (432; 70) and data and analytics (431; 66); quality assurance (148) and IT support (129) are the smallest. Pay is highest in software and IT project management ($72 an hour) and lowest in IT support ($32).
- **Openness varies most by bucket:** 44% of IT support listings show a sign of openness to career changers vs. 9% of IT project management; remote work is similar everywhere (17–23%, data and analytics highest).
- **The strongest paths are within tech:** IT support → networks and systems (279 people), networks and systems → IT support (199), IT support → security (163), developers → databases (142) and → data and analytics (137). From outside tech, the largest are business → networks and systems (31), arts and media → software (30), and computer science → software (75) by training.
- **Most paths are thin:** 78% of prior-career paths, 67% of training paths and 43% of within-tech paths rest on fewer than 10 people. The compass shows every count and marks thin paths, and `ML_Fit` should fall back to the bucket-level pattern when a user's exact path is thin.
- **Without likely ghosts** (3.1d), each bucket's median days open drops 1–3 days (security 26.8 → 24.2, software 24.1 → 22.2) and its open count about 10%: ghosts stretch timing a little, so both versions are kept.
- **Expected growth is still empty:** it fills once the official occupational projections are downloaded (queued). Saved for `ML_Fit`: `role_market.pkl` (8 buckets) and `compass_paths_2026.pkl` (223 paths).

## 4. EDA_SDA_2 final table

_To do after section 3:_ defend each decision with its result from section 3, then apply them all at once, at the end: the niche (computer and mathematical, kept role buckets), labelling manager-level listings (hidden per applicant in the app), the additional cleaning, each factor kept or dropped, and the ghost-listing decision. Record every step with its reason, in the same format as EDA_SDA_1 section 10 and 11. Save it as a separate file, so EDA_SDA_1's table stays unchanged.

## 5. Conclusion and next steps

_To do._ Next: `EDA_SDA_3.ipynb`, the text features, built on this notebook's niche table.